# QAIRT 2.42 Supertonic duration canary

Optional ahead-of-time context experiment for SM8850 / HTP-v81. This notebook does **not** train, profile, or execute on a phone. It downloads two public, checksum-pinned archives and runs only the 3.7 MB duration-predictor conversion in Google Colab. The raw ONNX phone path is the primary deployment route; this artifact is retained for startup/cache and reproducibility comparison.

Use an Ubuntu 22.04 x86_64 Colab runtime. Run the cells in order. Durable state and outputs live in Google Drive, so reconnecting and rerunning the final cells resumes safely.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess

EXPECTED = {
    'qairt242_colab_supertonic_duration_canary.py': 'bc39dc8fe71206177321c5fa9165cc3d7ea7d78ef3fbdb3b7e0add5f4a802ca2',
    'qairt242_colab_supertonic_duration_spec.json': '08e1425f1daba9626827dfce3b072f1191ff840c1e0f555743f39614546702ec',
}

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

print({'platform': platform.platform(), 'machine': platform.machine()})

In [ ]:
from google.colab import drive, files

drive.mount('/content/drive')
CONTROL = Path('/content/drive/MyDrive/lmf_qairt242_colab_supertonic_duration_v1/control')
WORKSPACE = Path('/content/drive/MyDrive/lmf_qairt242_colab_supertonic_duration_v1/workspace')
SCRATCH = Path('/content/qairt242_colab_scratch')
CONTROL.mkdir(parents=True, exist_ok=True)
WORKSPACE.mkdir(parents=True, exist_ok=True)

for name, expected_hash in EXPECTED.items():
    existing = CONTROL / name
    if existing.exists() and sha256_file(existing) != expected_hash:
        raise RuntimeError(f'Fail-closed: existing control file hash drift: {existing}')

missing = [name for name in EXPECTED if not (CONTROL / name).is_file()]
if missing:
    print('Upload exactly these repository files:', missing)
    uploaded = files.upload()
    for name in missing:
        if name not in uploaded:
            raise RuntimeError(f'Missing uploaded control file: {name}')
        data = uploaded[name]
        if sha256_bytes(data) != EXPECTED[name]:
            raise RuntimeError(f'Fail-closed: uploaded control hash mismatch: {name}')
        temporary = CONTROL / f'.{name}.partial'
        temporary.write_bytes(data)
        temporary.replace(CONTROL / name)

for name, expected_hash in EXPECTED.items():
    actual = sha256_file(CONTROL / name)
    assert actual == expected_hash, (name, actual)
print({'control': str(CONTROL), 'workspace': str(WORKSPACE), 'control_verified': True})

Install only the supported host toolchain. The canary itself creates a private Python 3.10 virtual environment and pins its QAIRT/ONNX dependencies. No GPU is needed.

In [ ]:
apt_env = dict(os.environ)
apt_env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['apt-get', 'update', '-qq'], check=True, env=apt_env)
subprocess.run([
    'apt-get', 'install', '-y', '--no-install-recommends',
    'python3.10', 'python3.10-venv', 'python3-distutils',
    'build-essential', 'clang-14', 'cmake', 'ninja-build', 'curl', 'unzip', 'bzip2', 'ca-certificates'
], check=True, env=apt_env)
python310 = shutil.which('python3.10')
if not python310:
    raise RuntimeError('Python 3.10 was not installed')
subprocess.run([python310, '--version'], check=True)
subprocess.run(['clang-14', '--version'], check=True)

Review the zero-write plan. It must show only `duration_predictor`, QAIRT 2.42, `soc_model=87`, and `dsp_arch=v81`.

In [ ]:
runner = CONTROL / 'qairt242_colab_supertonic_duration_canary.py'
spec = CONTROL / 'qairt242_colab_supertonic_duration_spec.json'
subprocess.run([python310, str(runner), 'plan', '--spec', str(spec)], check=True)

Run or resume the canary. The first run downloads about 1.55 GB of QAIRT plus the Supertonic archive, then performs the small graph conversion. A disconnect is safe: reconnect, remount Drive, validate/upload the same controls, reinstall host packages, and rerun this cell. Completed stages are reused only after their hashes pass.

In [ ]:
subprocess.run([
    python310, str(runner), 'run',
    '--spec', str(spec),
    '--workspace', str(WORKSPACE),
    '--scratch', str(SCRATCH),
], check=True)

Verify the durable result and print the portable bundle identity. `context_generated_unexecuted` is intentionally not a phone/NPU execution claim.

In [ ]:
subprocess.run([
    python310, str(runner), 'verify',
    '--spec', str(spec),
    '--workspace', str(WORKSPACE),
], check=True)
manifest = WORKSPACE / 'stages/deliverable/result_manifest.json'
bundle = WORKSPACE / 'stages/deliverable/qairt242_colab_result.tar.gz'
result = json.loads(manifest.read_text(encoding='utf-8'))
print({
    'status': result['status'],
    'context_sha256': result['context']['context_sha256'],
    'bundle': str(bundle),
    'bundle_bytes': bundle.stat().st_size,
    'bundle_sha256': sha256_file(bundle),
    'phone_execution_proven': result['phone_execution_proven'],
})
# Optional browser download after verification:
# files.download(str(bundle))